In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q datasets

import re
import json
import random
from collections import Counter
from datasets import load_dataset, Dataset
from urllib.parse import unquote

In [6]:
# Filter 1: Devanagari ratio
# Why: Catches English-leaking outputs (#14 alpaca, English-only samvaad convos).
# Logic: count Devanagari Unicode chars / total chars. If too low, output is mostly English.

def devanagari_ratio(text: str) -> float:
    """What fraction of the text is Devanagari script?"""
    if not text or len(text) == 0:
        return 0.0
    devanagari_chars = len(re.findall(r'[\u0900-\u097F]', text))
    return devanagari_chars / len(text)


# Filter 2: Length bounds
# Why: All your degenerate_long_output examples were >1000 chars (#5, #7, #17, #22, #30).
# All your 5s were <600 chars. Also drop very short outputs (likely lazy/identity).

def length_ok(text: str, min_chars: int = 20, max_chars: int = 700) -> bool:
    """Is the text in the acceptable length range?"""
    return min_chars <= len(text) <= max_chars


# Filter 3: Identity check
# Why: Catches #4 where output was literally the input. Useless training signal.

def is_identity(input_text: str, output_text: str, similarity_threshold: float = 0.95) -> bool:
    """Is the output essentially the same as the input?"""
    if not input_text or not output_text:
        return False
    # Simple character-level overlap; for production, use difflib or rapidfuzz
    # but this is fast and good enough
    inp = input_text.strip().lower()
    out = output_text.strip().lower()
    if inp == out:
        return True
    # Check if output is just input with trivial changes
    if len(inp) > 0 and len(out) > 0:
        overlap = len(set(inp.split()) & set(out.split()))
        max_words = max(len(inp.split()), len(out.split()))
        if max_words > 0 and overlap / max_words >= similarity_threshold:
            return True
    return False


# Filter 4: Repetition detection
# Why: Catches degenerate outputs where same phrase repeats (#17 "अद्भुत" x4, #30 "महानता" x4).
# Logic: find n-grams (sequences of n words) and check if any repeat too often.

def has_repetition_loop(text: str, n: int = 4, max_repeats: int = 3) -> bool:
    """Does any n-word sequence repeat more than max_repeats times?"""
    words = text.split()
    if len(words) < n * 2:
        return False
    ngrams = [' '.join(words[i:i+n]) for i in range(len(words) - n + 1)]
    counts = Counter(ngrams)
    most_common_count = counts.most_common(1)[0][1] if counts else 0
    return most_common_count >= max_repeats


# Filter 5: AI refusal pattern
# Why: Catches #8 "मैं एक एआई भाषा मॉडल हूँ" type over-refusals.

AI_REFUSAL_PATTERNS = [
    "मैं एक एआई",
    "एक भाषा मॉडल",
    "एक भाषा मॉडल के रूप में",
    "मैं एक भाषा मॉडल",
    "मेरे पास कोई व्यक्तिगत",
    "I am an AI",
    "as an AI language model",
    "I'm an AI",
]

def is_ai_refusal(text: str) -> bool:
    """Does the text start with an AI-identity refusal pattern?"""
    text_start = text.strip()[:100].lower()
    return any(pattern.lower() in text_start for pattern in AI_REFUSAL_PATTERNS)

# Filter 6 (improved): Harmful content with multilingual variants
# Why: Hindi text mixes scripts freely. Same concept may appear in 3-4 forms.

HARMFUL_PATTERNS = [
    # SQL injection variants
    "sql injection", "sql इंजेक्शन", "एसक्यूएल इंजेक्शन", "एस क्यू एल इंजेक्शन",

    # Hacking variants
    "हैक करें", "हैक करना", "hack into", "hacking into",
    "अनधिकृत पहुंच", "unauthorized access",

    # Exploit / vulnerability
    "exploit vulnerability", "exploit a vulnerability",
    "कमजोरी का शोषण", "कमज़ोरी का शोषण",
    "हमले वेक्टर", "attack vector",

    # Phishing
    "phishing", "फ़िशिंग", "फिशिंग",

    # Malware
    "malware", "मैलवेयर", "वायरस बनाएं", "create a virus",

    # Other
    "bypass authentication", "प्रमाणीकरण को दरकिनार",
    "dos attack", "ddos attack", "डीडीओएस",
]

def has_harmful_content(instruction: str, output: str) -> bool:
    """Check if the example contains content involving cyber attacks, hacking, or similar harms.

    Note: This is a deliberately conservative keyword filter. It will produce some
    false positives (e.g. educational discussion of security concepts) but for
    instruction-tuning data, erring on the side of dropping is safer than including.
    """
    combined = (instruction + " " + output).lower()
    return any(pattern.lower() in combined for pattern in HARMFUL_PATTERNS)

In [7]:
# Test cases from your Week 1 analysis
test_cases = [
    # (description, instruction, output, expected_to_filter_out, reason)
    ("alpaca #1 - SQL injection (should drop)",
     "एक संभावित हमले वेक्टर का निर्माण करें",
     "एक हमलावर SQL इंजेक्शन का उपयोग कर सकता है",
     True, "harmful_content"),

    ("alpaca #4 - identity output (should drop)",
     "उसे फिल्म बहुत पसंद आई।",
     "उसे फिल्म बहुत पसंद आई।",
     True, "identity"),

    ("alpaca #6 - clean short output (should keep)",
     "एक दिन का वर्णन करने वाले तीन विशेषण लिखिए।",
     "धूप, आरामदायक, उत्पादक।",
     False, None),

    ("alpaca #14 - english output (should drop)",
     "जावास्क्रिप्ट फ़ंक्शन लिखें",
     "Here is an example solution: function squareSum(n, arr) { return sum * sum; }",
     True, "low_devanagari"),

    ("alpaca #8 - AI refusal (should drop)",
     "अपने शहर में 3 जगहों का नाम बताइए।",
     "मैं एक एआई भाषा मॉडल हूँ, इसलिए मेरे पास कोई विशिष्ट शहर नहीं है",
     True, "ai_refusal"),
]

def apply_all_filters(instruction: str, output: str) -> tuple[bool, str]:
    """Returns (should_keep, reason_if_dropped)"""
    if has_harmful_content(instruction, output):
        return False, "harmful_content"
    if is_identity(instruction, output):
        return False, "identity"
    if not length_ok(output):
        return False, "length"
    if devanagari_ratio(output) < 0.30:
        return False, "low_devanagari"
    if has_repetition_loop(output):
        return False, "repetition_loop"
    if is_ai_refusal(output):
        return False, "ai_refusal"
    return True, None


# Run tests
print("Running filter unit tests:\n")
for desc, instruction, output, should_drop, expected_reason in test_cases:
    should_keep, actual_reason = apply_all_filters(instruction, output)
    expected_keep = not should_drop
    passed = (should_keep == expected_keep)
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"{status}  {desc}")
    if not passed:
        print(f"        Expected: keep={expected_keep}, reason={expected_reason}")
        print(f"        Got:      keep={should_keep}, reason={actual_reason}")

Running filter unit tests:

✓ PASS  alpaca #1 - SQL injection (should drop)
✓ PASS  alpaca #4 - identity output (should drop)
✓ PASS  alpaca #6 - clean short output (should keep)
✓ PASS  alpaca #14 - english output (should drop)
✓ PASS  alpaca #8 - AI refusal (should drop)


In [8]:
from datasets import load_dataset

ds1 = load_dataset("iamshnoo/alpaca-cleaned-hindi", split="train")
print(f"Loaded {len(ds1):,} examples")
print(f"Columns: {ds1.column_names}")
print(f"\nFirst example:\n{ds1[0]}")

README.md:   0%|          | 0.00/497 [00:00<?, ?B/s]

data/train-00000-of-00001-9c3cdcdad5ba72(…):   0%|          | 0.00/31.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Loaded 51,760 examples
Columns: ['input', 'instruction', 'output']

First example:
{'input': '', 'instruction': 'स्वस्थ रहने के लिए तीन सुझाव दें।', 'output': '1. संतुलित और पौष्टिक आहार खाएं: अपने भोजन में विभिन्न प्रकार के फल और सब्जियां, दुबला प्रोटीन, पूरे अनाज और स्वस्थ वसा शामिल हों। इससे आपके शरीर को आवश्यक पोषक तत्वों से भरपूर होता है ताकि वह अपने कामकाज को सबसे अच्छा कर सके और इससे पुरानी बीमारियों को रोकने में मदद मिल सकती है। 2. नियमित शारीरिक गतिविधि करें: मजबूत हड्डियों, मांसपेशियों और हृदय स्वास्थ्य को बनाए रखने के लिए व्यायाम महत्वपूर्ण है। हर हफ्ते कम से कम 150 मिनट का मध्यम एरोबिक व्यायाम या 75 मिनट का जोरदार व्यायाम करना लक्ष्य रखें। 3. पर्याप्त नींद लें: पर्याप्त गुणवत्ता की नींद लेना शारीरिक और मानसिक भलाई के लिए महत्वपूर्ण है। यह मनोदशा को विनियमित करने, संज्ञानात्मक कार्य को बेहतर बनाने और स्वस्थ विकास और प्रतिरक्षा कार्य को बढ़ावा देने में मदद करता है। हर रात 7-9 घंटे की नींद लेना लक्ष्य रखें।'}


In [9]:
from collections import Counter
import json

# Track everything
drop_reasons = Counter()
kept_examples = []

# Apply filters to every example
for i, ex in enumerate(ds1):
    instruction = ex.get('instruction', '') or ''
    input_text = ex.get('input', '') or ''
    output = ex.get('output', '') or ''

    # Combine instruction + input as the "user prompt" for identity checking
    full_input = (instruction + ' ' + input_text).strip()

    # Run filters
    should_keep, reason = apply_all_filters(full_input, output)

    if should_keep:
        kept_examples.append({
            'instruction': instruction,
            'input': input_text,
            'output': output,
            'source': 'alpaca-hindi'
        })
    else:
        drop_reasons[reason] += 1

    # Progress indicator every 5000
    if (i + 1) % 5000 == 0:
        print(f"  Processed {i+1:,} / {len(ds1):,} ...")

print("\n" + "=" * 50)
print("CLEANING REPORT — iamshnoo/alpaca-cleaned-hindi")
print("=" * 50)
print(f"Original: {len(ds1):,} examples")
total_dropped = sum(drop_reasons.values())
print(f"Dropped:  {total_dropped:,} ({100*total_dropped/len(ds1):.1f}%)")
for reason, count in drop_reasons.most_common():
    print(f"  - {reason:18s}: {count:,}")
print(f"Kept:     {len(kept_examples):,} ({100*len(kept_examples)/len(ds1):.1f}%)")

  Processed 5,000 / 51,760 ...
  Processed 10,000 / 51,760 ...
  Processed 15,000 / 51,760 ...
  Processed 20,000 / 51,760 ...
  Processed 25,000 / 51,760 ...
  Processed 30,000 / 51,760 ...
  Processed 35,000 / 51,760 ...
  Processed 40,000 / 51,760 ...
  Processed 45,000 / 51,760 ...
  Processed 50,000 / 51,760 ...

CLEANING REPORT — iamshnoo/alpaca-cleaned-hindi
Original: 51,760 examples
Dropped:  23,492 (45.4%)
  - length            : 21,663
  - low_devanagari    : 772
  - repetition_loop   : 649
  - harmful_content   : 215
  - ai_refusal        : 192
  - identity          : 1
Kept:     28,268 (54.6%)


In [10]:
# Step 4: Save
import json
output_path = '/kaggle/working/dataset1_alpaca_cleaned.jsonl'
with open(output_path, 'w', encoding='utf-8') as f:
    for ex in kept_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')
print(f"Saved {len(kept_examples):,} examples to {output_path}")

Saved 28,268 examples to /kaggle/working/dataset1_alpaca_cleaned.jsonl


In [11]:
# Step 5: Sanity check
import random
random.seed(42)
print("Random samples of KEPT examples:\n")
for ex in random.sample(kept_examples, 5):
    print(f"INSTRUCTION: {ex['instruction'][:100]}")
    print(f"OUTPUT ({len(ex['output'])} chars): {ex['output'][:200]}")
    print("-" * 40)

Random samples of KEPT examples:

INSTRUCTION: आधुनिक भाषा का उपयोग करके निम्नलिखित वाक्य को अद्यतन करें:
OUTPUT (64 chars): वह उसकी प्रतिक्रिया का इंतजार कर रहा था, उसकी सांसें रोक रहा था।
----------------------------------------
INSTRUCTION: दिए गए निर्देश के लिए उपयुक्त इनपुट प्रदान करें।
OUTPUT (89 chars): कृपया अंग्रेजी में वह वाक्य प्रदान करें जिसे आप चाहते हैं कि मैं स्पेनिश में अनुवाद करूं।
----------------------------------------
INSTRUCTION: ब्रदर्स ग्रिम्स के बारे में इस लेख का सारांश दें।
OUTPUT (419 chars): ब्रदर्स ग्रिम्, जेकब और विल्हेम, जर्मन लोक कथाकार और भाषाविद थे, जो अपनी प्रसिद्ध परी कथाओं के संग्रह के लिए जाने जाते थे, जिसने लोक कथा के आधुनिक अध्ययन को जन्म दिया; उन्होंने लोक संगीत और साहित्य के
----------------------------------------
INSTRUCTION: निम्नलिखित कविता में इस्तेमाल की गई एक मिसाल दीजिए।
OUTPUT (171 chars): कविता में एक तुलना का प्रयोग किया गया है "**रात में चमकते सितारों की तरह**". यह कुछ सुंदर की तुलना करता है जिसे लेखक की आंखें रात के आकाश में चमकते

## Day 2: Apply Cleaning Pipeline to Dataset 1

**Dataset:** `iamshnoo/alpaca-cleaned-hindi`
**Goal:** Apply all 6 filter functions from Day 1 to the full ~52K dataset, measure drop rates by filter, and save the cleaned subset to disk.

### Why dataset 1 first?
This is the lowest-quality dataset of the three (Week 1 average score: 2.93/5) and has the most diverse failure modes. Cleaning it first stress-tests the filter pipeline.

## Cleaning Results: iamshnoo/alpaca-cleaned-hindi

| Metric | Value |
|--------|-------|
| Original | 51,760 |
| Dropped  | 23,492 (45.4%) |
| Kept     | 28,268 (54.6%) |

### Drop breakdown by filter
| Filter | Examples removed | % of dropped |
|--------|------------------|--------------|
| length            | 21,663 | 92.2% |
| low_devanagari    | 772    | 3.3% |
| repetition_loop   | 649    | 2.8% |
| harmful_content   | 215    | 0.9% |
| ai_refusal        | 192    | 0.8% |
| identity          | 1      | 0.0% |

### Observations
- **Length filter dominated:** 92% of removed examples were dropped purely for length (<20 or >700 chars). This confirms the Week 1 hypothesis that long outputs in this dataset reliably degenerate into repetition or incoherence.
- **English leakage at scale:** 772 examples (~1.5%) had outputs in mostly English despite Hindi instructions. Mostly code/technical content where the source generation model defaulted to English.
- **Repetition loops:** 649 examples (~1.3%) contained 4-gram repetitions, matching the pattern seen in Week 1 samples #17 and #30.
- **Identity filter caught only 1:** The 95% word-overlap threshold may be too strict. The single-occurrence rate suggests true identity outputs are rare in this dataset — most laziness manifests as short or repetitive outputs rather than literal copies.
- **45.4% drop rate aligns with manual analysis:** In Week 1, 44% of 30 sampled examples scored 1-2. The pipeline's automated drop rate (45.4%) is consistent with that human assessment.

### Manual verification of cleaned dataset 1

Random sample of 5 kept examples (seed=42) reviewed manually:
- **4 of 5** are clean, high-quality Hindi instruction-following examples (modernization rewrite, factual summary, literary analysis, technical explanation)
- **1 of 5** is borderline — output asks user for clarification rather than performing the task

### Known residual issue: clarification-request outputs
Outputs where the model deflects by asking for more input pass through current filters (e.g., "कृपया अंग्रेजी में वह वाक्य प्रदान करें..."). Estimated <5% of kept examples based on this sample. 

This is a milder version of the AI refusal pattern but doesn't use refusal-keyword phrases, so the existing filter misses it. Acceptable for this project's scope; a future improvement would add a "deflection detection" filter or use a small classifier model trained on refusal/deflection examples.

### Output
Saved 28,268 cleaned examples to `dataset1_alpaca_cleaned.jsonl` (JSONL format, one example per line, UTF-8 encoded).

**Schema of each line:**
```json
{
  "instruction": "...",
  "input": "...",
  "output": "...",
  "source": "alpaca-hindi"
}
```

The `source` tag will be used in Day 5 to track contributions when datasets are combined.